# 06 QR 分解：Gram-Schmidt、Householder 与 Givens

QR 分解写作

$$
A=QR,
$$

其中 $Q$ 的列向量正交，$R$ 为上三角矩阵。正交变换保持二范数，因此 QR 在最小二乘和稳定线性代数中非常重要。


In [ ]:
import pathlib
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch06_direct_linear_systems"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import classical_gram_schmidt, givens_qr, householder_qr, modified_gram_schmidt, orthogonality_error, qr_solve


## 经典 Gram-Schmidt

对列向量 $\boldsymbol{a}_j$，经典 Gram-Schmidt 一次性减去它在已有正交向量上的投影：

$$
\boldsymbol{v}_j=\boldsymbol{a}_j-\sum_{i<j}(\boldsymbol{q}_i^T\boldsymbol{a}_j)\boldsymbol{q}_i.
$$


In [ ]:
def teaching_classical_gs(A):
    A = np.asarray(A, dtype=float)
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            R[i, j] = Q[:, i] @ A[:, j]
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    return Q, R

A = np.array([[1.0, 1.0, 1.0], [1.0, 1.0 + 1e-8, 2.0], [1.0, 2.0, 3.0], [1.0, 3.0, 5.0]])
Q, R = teaching_classical_gs(A)
print("重构误差:", np.linalg.norm(Q @ R - A))
print("正交性误差:", np.linalg.norm(Q.T @ Q - np.eye(Q.shape[1])))


## 修正 Gram-Schmidt

修正 Gram-Schmidt 每次都从当前剩余向量中减去投影，在浮点运算下通常比经典形式更好地保持正交性。


In [ ]:
Q_c, R_c = classical_gram_schmidt(A)
Q_m, R_m = modified_gram_schmidt(A)
print("经典 GS 正交性误差:", orthogonality_error(Q_c))
print("修正 GS 正交性误差:", orthogonality_error(Q_m))
print("修正 GS 重构误差:", np.linalg.norm(Q_m @ R_m - A))


## Householder 反射

Householder 反射

$$
H=I-2\boldsymbol{v}\boldsymbol{v}^T
$$

可以把一个向量映射到坐标轴方向，从而一次消去一列主元下方所有元素。下面完整展示第一步反射向量和矩阵更新。


In [ ]:
x = A[:, 0]
norm_x = np.linalg.norm(x)
v = x.copy()
v[0] += norm_x
v = v / np.linalg.norm(v)
H = np.eye(A.shape[0]) - 2.0 * np.outer(v, v)
print("第一步 Householder 向量:", v)
print("H @ 第一列:", H @ x)
print("第一步更新后的矩阵:\n", H @ A)

Q_h, R_h = householder_qr(A)
print("Householder 重构误差:", np.linalg.norm(Q_h @ R_h - A))
print("Householder 正交性误差:", orthogonality_error(Q_h))


## Givens 旋转

Givens 旋转只消去一个元素：

$$
\begin{bmatrix}c&s\\-s&c\end{bmatrix}
\begin{bmatrix}a\\b\end{bmatrix}
=
\begin{bmatrix}\sqrt{a^2+b^2}\\0\end{bmatrix}.
$$

它特别适合稀疏矩阵和增量更新。本章给出基础实现，高性能稀疏 QR 属于后续拓展。


In [ ]:
G_example = np.array([[3.0, 1.0], [4.0, 2.0]])
a, b = G_example[0, 0], G_example[1, 0]
r = np.hypot(a, b)
c, s = a / r, b / r
G = np.array([[c, s], [-s, c]])
print("旋转参数 c, s:", c, s)
print("旋转后矩阵:\n", G @ G_example)

Q_g, R_g = givens_qr(A)
print("Givens 重构误差:", np.linalg.norm(Q_g @ R_g - A))
print("Givens 正交性误差:", orthogonality_error(Q_g))


## 用 QR 求解方阵系统

若 $A=QR$，则

$$
R\boldsymbol{x}=Q^T\boldsymbol{b}.
$$


In [ ]:
A_square = np.array([[3.0, 1.0, -1.0], [2.0, 4.0, 1.0], [-1.0, 2.0, 5.0]])
x_exact = np.array([1.0, -2.0, 0.5])
b = A_square @ x_exact
Q, R = householder_qr(A_square)
x = qr_solve(Q, R, b)
print("QR 求解结果:", x)
print("与精确解差:", np.linalg.norm(x - x_exact))


## 小结

* Gram-Schmidt 直观，但经典形式在近线性相关列上容易丢失正交性。
* 修正 Gram-Schmidt 通常比经典形式更稳定。
* Householder 是稠密 QR 的常用稳定方法。
* Givens 一次消去一个元素，适合稀疏和增量问题；完整高性能实现待完善。
* QR 的验证应同时看 $QR\approx A$ 和 $Q^TQ\approx I$。
